# 05 - Ingestão Silver: Limpeza, Tratamento e Validação de Regras do Micro-Lote
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas de Escopo:** `ecommerce_produtos` e `ecommerce_categorias`  
**Branch:** `feat/squad2-lucas_zaiden`  

### Objetivo da Task (Sprint 2 - Task 2):
1. **Leitura Incremental da Camada Bronze:** Consumir os micro-lotes recém-ingeridos via Delta Streaming com `.trigger(availableNow=True)`.
2. **Auditoria Silver:** Adicionar a coluna de rastreabilidade `'silver_processed_at'` com `current_timestamp()`.
3. **Limpeza e Padronização:** Aplicar `trim` nos campos textuais e casting estrito de tipos (`preco_lista`, `is_ativo`).
4. **Validação das Regras Técnicas (Padrão Quarentena):**
   * `ecommerce_produtos`:
     * *Técnica 1:* Comprimento do SKU entre 5 e 60 caracteres (`5 < len(sku) < 60`).
     * *Técnica 2:* Faixa operacional de preço (`0 < preco_lista < 5000`).
     * *Técnica 3:* `is_ativo` não nulo e booleano.
   * `ecommerce_categorias`:
     * *Técnica 1:* `id_categoria` e `nome_categoria` obrigatórios (não nulos nem vazios).
   * Registros que violarem as regras são encaminhados para a **Quarentena** com o motivo em `quarantine_reason`.
5. **Monitoramento e Alertas de Negócio:**
   * *Negócio 4 (Produtos):* Alertar se o micro-lote adicionar `> 50` novos SKUs (possível carga de teste).
   * *Negócio 5 (Produtos):* Alertar criticamente se produto ativo tiver `preco_lista <= 0`.
   * *Negócio 2 (Categorias):* Alertar se a contagem de categorias raiz (`id_categoria_pai IS NULL`) for alterada.
6. **Persistência em Delta Silver (`mode: append`):** Salvar os registros válidos no container `squad2/grupo1/silver/` em formato Delta e registrar no Databricks Metastore.

## 1. Carregamento de Credenciais e Configurações de Conexão

In [0]:
import os
from dotenv import load_dotenv, find_dotenv

# Carregamento automático do arquivo .env com override
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Configurações OAuth do Service Principal na sessão Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}
for k, v in adls_options.items():
    spark.conf.set(k, v)

# Definição dos caminhos com isolamento de namespace no prefixo /grupo1/
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1"

caminhos = {
    "bronze_produtos": f"{base_squad}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad}/bronze/ecommerce_categorias",
    "silver_produtos": f"{base_squad}/silver/ecommerce_produtos",
    "silver_categorias": f"{base_squad}/silver/ecommerce_categorias",
    "quarantine_produtos": f"{base_squad}/quarantine/ecommerce_produtos",
    "quarantine_categorias": f"{base_squad}/quarantine/ecommerce_categorias",
    "checkpoint_silver_produtos": f"{base_squad}/checkpoints/silver_produtos",
    "checkpoint_silver_categorias": f"{base_squad}/checkpoints/silver_categorias"
}

print("Caminhos configurados para a Camada Silver:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")

## 2. Funções de Tratamento, Validação e Quarentena de Micro-Lote
Implementam o processamento modular para separar registros válidos dos reprovados e emitir alertas de negócio em tempo real.

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, trim, length, when, lit, concat_ws, array, expr, count, countDistinct
)
from pyspark.sql.types import DoubleType, BooleanType

# Processador do micro-lote de Produtos
def processar_silver_produtos(df_batch, batch_id):
    total = df_batch.count()
    if total == 0:
        return

    print(f"\n===================================================")
    print(f">>> [Micro-Lote {batch_id}] Processando Silver: ecommerce_produtos ({total} registros)")

    # 1. Auditoria Silver, Limpeza de Strings e Tipagem Estrita
    df_cleaned = (df_batch
        .withColumn("silver_processed_at", current_timestamp())
        .withColumn("sku", trim(col("sku")))
        .withColumn("nome_produto", trim(col("nome_produto")))
        .withColumn("descricao", trim(col("descricao")))
        .withColumn("id_categoria", trim(col("id_categoria")))
        .withColumn("unidade_medida", trim(col("unidade_medida")))
        .withColumn("nome_marca", trim(col("nome_marca")))
        .withColumn("preco_lista", col("preco_lista").cast(DoubleType()))
        .withColumn("is_ativo", col("is_ativo").cast(BooleanType())))

    # 2. Regras Técnicas (Validações de Quarentena)
    # Técnica 1: 5 < len(sku) < 60
    cond_sku_valido = (length(col("sku")) > 5) & (length(col("sku")) < 60)
    # Técnica 2: 0 < preco_lista < 5000
    cond_preco_valido = (col("preco_lista") > 0) & (col("preco_lista") < 5000)
    # Técnica 3: is_ativo não nulo
    cond_ativo_valido = col("is_ativo").isNotNull()

    # Identificar motivos de falha para a quarentena
    df_tagged = df_cleaned.withColumn(
        "quarantine_reason",
        concat_ws("; ",
            when(~cond_sku_valido, lit("FALHA_TECNICA_1: Comprimento do SKU fora da faixa (5 a 60 caracteres)")),
            when(~cond_preco_valido, lit("FALHA_TECNICA_2: Preço de lista fora da faixa permitida (0 a 5000)")),
            when(~cond_ativo_valido, lit("FALHA_TECNICA_3: Campo is_ativo nulo ou tipo inválido"))
        )
    ).withColumn("quarantined_at", current_timestamp())

    # Separação entre Válidos e Quarentena
    df_validos = df_tagged.filter(col("quarantine_reason") == "").drop("quarantine_reason", "quarantined_at")
    df_quarentena = df_tagged.filter(col("quarantine_reason") != "")

    total_validos = df_validos.count()
    total_quarentena = df_quarentena.count()
    print(f"    -> Registros Válidos:    {total_validos}")
    print(f"    -> Registros Quarentena: {total_quarentena}")

    # 3. Regras de Negócio e Alertas Operacionais
    # Negócio 5: Alertar se produto ativo receber preco_lista <= 0
    anomalias_preco = df_cleaned.filter((col("is_ativo") == True) & ((col("preco_lista") <= 0) | col("preco_lista").isNull())).count()
    if anomalias_preco > 0:
        print(f"    🚨 [ALERTA CRÍTICO DE NEGÓCIO - REGRA 5]: Detectados {anomalias_preco} produtos ATIVOS com preço <= 0!")

    # Negócio 4: Monitorar número de novos SKUs no micro-lote
    # Verificar se a Silver já existe para comparar SKUs novos vs existentes
    try:
        df_silver_existente = spark.read.format("delta").load(caminhos["silver_produtos"])
        skus_novos = df_validos.join(df_silver_existente, on="sku", how="left_anti").select(countDistinct("sku")).first()[0]
    except Exception:
        # Primeira carga
        skus_novos = df_validos.select(countDistinct("sku")).first()[0]

    print(f"    📊 [KPI NEGÓCIO - REGRA 4]: Novos SKUs adicionados neste lote: {skus_novos}")
    if skus_novos > 50:
        print(f"    ⚠️ [ALERTA DE NEGÓCIO - REGRA 4]: Lote com mais de 50 novos SKUs ({skus_novos}). Verificar se é carga de teste!")

    # 4. Gravação Delta Silver (modo append conforme especificado na task)
    if total_validos > 0:
        (df_validos.write
            .format("delta")
            .mode("append")
            .save(caminhos["silver_produtos"]))
        print(f"    -> {total_validos} registros gravados com sucesso na Delta Silver.")

    # 5. Gravação da Quarentena (caso existam reprovados)
    if total_quarentena > 0:
        (df_quarentena.write
            .format("delta")
            .mode("append")
            .save(caminhos["quarantine_produtos"]))
        print(f"    -> {total_quarentena} registros isolados gravados na Quarentena.")

    print(f"<<< [Micro-Lote {batch_id}] Concluído com sucesso.")
    print(f"===================================================")


# Processador do micro-lote de Categorias
def processar_silver_categorias(df_batch, batch_id):
    total = df_batch.count()
    if total == 0:
        return

    print(f"\n===================================================")
    print(f">>> [Micro-Lote {batch_id}] Processando Silver: ecommerce_categorias ({total} registros)")

    # 1. Auditoria Silver e Limpeza de Strings
    df_cleaned = (df_batch
        .withColumn("silver_processed_at", current_timestamp())
        .withColumn("id_categoria", trim(col("id_categoria")))
        .withColumn("nome_categoria", trim(col("nome_categoria")))
        .withColumn("id_categoria_pai", trim(col("id_categoria_pai")))
        .withColumn("nome_categoria_pai", trim(col("nome_categoria_pai")))
        .withColumn("tipo_categoria", trim(col("tipo_categoria"))))

    # 2. Regras Técnicas (Técnica 1: id_categoria e nome_categoria obrigatórios)
    cond_id_valido = col("id_categoria").isNotNull() & (length(col("id_categoria")) > 0)
    cond_nome_valido = col("nome_categoria").isNotNull() & (length(col("nome_categoria")) > 0)

    df_tagged = df_cleaned.withColumn(
        "quarantine_reason",
        concat_ws("; ",
            when(~cond_id_valido, lit("FALHA_TECNICA_1: id_categoria nulo ou vazio")),
            when(~cond_nome_valido, lit("FALHA_TECNICA_1: nome_categoria nulo ou vazio"))
        )
    ).withColumn("quarantined_at", current_timestamp())

    df_validos = df_tagged.filter(col("quarantine_reason") == "").drop("quarantine_reason", "quarantined_at")
    df_quarentena = df_tagged.filter(col("quarantine_reason") != "")

    total_validos = df_validos.count()
    total_quarentena = df_quarentena.count()
    print(f"    -> Registros Válidos:    {total_validos}")
    print(f"    -> Registros Quarentena: {total_quarentena}")

    # 3. Regra de Negócio 2: Monitorar categorias raiz (id_categoria_pai IS NULL)
    cat_raiz_lote = df_validos.filter(col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "")).select(countDistinct("id_categoria")).first()[0]
    print(f"    🌳 [GOVERNANÇA NEGÓCIO - REGRA 2]: Categorias raiz identificadas no lote: {cat_raiz_lote}")

    try:
        df_silver_cat = spark.read.format("delta").load(caminhos["silver_categorias"])
        cat_raiz_existentes = df_silver_cat.filter(col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "")).select(countDistinct("id_categoria")).first()[0]
        if cat_raiz_lote != cat_raiz_existentes and cat_raiz_existentes > 0:
            print(f"    ⚠️ [ALERTA DE NEGÓCIO - REGRA 2]: Número total de categorias raiz mudou! (Anterior: {cat_raiz_existentes} vs Lote: {cat_raiz_lote}). Impacto na navegação do e-commerce!")
    except Exception:
        pass

    # 4. Gravação Delta Silver (modo append)
    if total_validos > 0:
        (df_validos.write
            .format("delta")
            .mode("append")
            .save(caminhos["silver_categorias"]))
        print(f"    -> {total_validos} registros gravados com sucesso na Delta Silver.")

    # 5. Gravação da Quarentena (caso existam reprovados)
    if total_quarentena > 0:
        (df_quarentena.write
            .format("delta")
            .mode("append")
            .save(caminhos["quarantine_categorias"]))
        print(f"    -> {total_quarentena} registros isolados gravados na Quarentena.")

    print(f"<<< [Micro-Lote {batch_id}] Concluído com sucesso.")
    print(f"===================================================")

print("Processadores Silver configurados com sucesso.")

## 3. Execução do Pipeline Streaming Silver: `ecommerce_produtos`
Consome a camada Bronze de forma incremental via Delta Streaming e grava na Silver em modo `append`.

In [0]:
print("Iniciando consumo incremental Bronze -> Silver para 'ecommerce_produtos'...")

# Leitura Delta streaming a partir da Bronze
stream_bronze_produtos = (spark.readStream
    .format("delta")
    .load(caminhos["bronze_produtos"]))

# Processamento com trigger availableNow e foreachBatch
query_silver_produtos = (stream_bronze_produtos.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", caminhos["checkpoint_silver_produtos"])
    .trigger(availableNow=True)
    .foreachBatch(processar_silver_produtos)
    .start())

query_silver_produtos.awaitTermination()
print("Processamento Silver de 'ecommerce_produtos' concluído com sucesso!")

## 4. Execução do Pipeline Streaming Silver: `ecommerce_categorias`
Consome a camada Bronze de categorias de forma incremental e grava na Silver em modo `append`.

In [0]:
print("Iniciando consumo incremental Bronze -> Silver para 'ecommerce_categorias'...")

# Leitura Delta streaming a partir da Bronze
stream_bronze_categorias = (spark.readStream
    .format("delta")
    .load(caminhos["bronze_categorias"]))

# Processamento com trigger availableNow e foreachBatch
query_silver_categorias = (stream_bronze_categorias.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", caminhos["checkpoint_silver_categorias"])
    .trigger(availableNow=True)
    .foreachBatch(processar_silver_categorias)
    .start())

query_silver_categorias.awaitTermination()
print("Processamento Silver de 'ecommerce_categorias' concluído com sucesso!")

## 5. Criação e Mapeamento das Tabelas Externas no Databricks Metastore
Mapeia as tabelas Silver e Quarentena sob o schema `squad2` no catálogo do Databricks.

In [0]:
# Garantir existência do schema squad2
spark.sql("CREATE SCHEMA IF NOT EXISTS squad2")

# 1. Tabela Externa Silver de Produtos
spark.sql(f"""
CREATE TABLE IF NOT EXISTS squad2.silver_ecommerce_produtos
USING DELTA
LOCATION '{caminhos["silver_produtos"]}'
""")

# 2. Tabela Externa Silver de Categorias
spark.sql(f"""
CREATE TABLE IF NOT EXISTS squad2.silver_ecommerce_categorias
USING DELTA
LOCATION '{caminhos["silver_categorias"]}'
""")

# 3. Tabela Externa de Quarentena de Produtos (se existir)
try:
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.quarantine_ecommerce_produtos
    USING DELTA
    LOCATION '{caminhos["quarantine_produtos"]}'
    """)
except Exception:
    pass

# 4. Tabela Externa de Quarentena de Categorias (se existir)
try:
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.quarantine_ecommerce_categorias
    USING DELTA
    LOCATION '{caminhos["quarantine_categorias"]}'
    """)
except Exception:
    pass

print("Tabelas externas Silver e Quarentena registradas no Databricks Metastore!")

## 6. Auditoria de Qualidade dos Dados na Camada Silver

In [0]:
print("=== Relatório de Auditoria Silver & Quarentena ===\n")

# Leitura das tabelas Silver consolidadas
df_silver_prod = spark.read.format("delta").load(caminhos["silver_produtos"])
df_silver_cat = spark.read.format("delta").load(caminhos["silver_categorias"])

total_prod = df_silver_prod.count()
unicos_prod = df_silver_prod.select(countDistinct("sku")).first()[0]
total_cat = df_silver_cat.count()
unicos_cat = df_silver_cat.select(countDistinct("id_categoria")).first()[0]

print(f"Produtos na Silver:   {total_prod} registros ({unicos_prod} SKUs únicos)")
print(f"Categorias na Silver: {total_cat} registros ({unicos_cat} Categorias únicas)")

# Verificação de Quarentena
try:
    df_quar_prod = spark.read.format("delta").load(caminhos["quarantine_produtos"])
    print(f"Produtos na Quarentena: {df_quar_prod.count()} registros")
    print("\nAmostra de Produtos Reprovados:")
    display(df_quar_prod.select("sku", "preco_lista", "is_ativo", "quarantine_reason").limit(5))
except Exception:
    print("Produtos na Quarentena: 0 registros (100% de conformidade técnica)")

# Amostras dos Dados Higienizados
print("\n--- Amostra Silver: Produtos higienizados com 'silver_processed_at' ---")
display(df_silver_prod.select("sku", "nome_produto", "preco_lista", "is_ativo", "silver_processed_at").limit(5))

print("\n--- Amostra Silver: Categorias higienizadas com 'silver_processed_at' ---")
display(df_silver_cat.select("id_categoria", "nome_categoria", "tipo_categoria", "silver_processed_at").limit(5))